# P-EAGLE with a narrower feed-forward layer

**Question (RQ1).** If we make the drafter's feed-forward layers narrower (75% and 50% of the
original width), does the drafter keep enough of its accepted tokens to stay fast?

**What this notebook does, in five steps.** One A100. Plain PyTorch, so every line is visible.

| Step | What it does | Time on an A100 (full settings) |
|---|---|---|
| 1. Data | Llama-3.1-8B writes answers to 3,000 chat prompts | about 25 min |
| 2. Draft vocabulary | Keeps the 32,000 most frequent answer tokens | seconds |
| 3. Train | Trains the same P-EAGLE-style drafter at 100%, 75% and 50% width | about 30 min each |
| 4. Check | The drafter sees the same inputs in training and in decoding; speculative = standard output | 1 min |
| 5. Measure | Tokens per second, accepted tokens per cycle, time per part of a cycle | about 30 min |

**Fair comparison.** All three drafters use the same data, the same order, the same number of
updates and the same draft vocabulary. All measurements use the same GPU, prompts, draft length
(K = 8), greedy decoding and one request at a time.

**First run.** Keep `PILOT = True`. Everything finishes in about 15 minutes and shows that the code
works. Then set `PILOT = False` and run all cells again. Finished work is saved under `OUT` on the
Insomnia filesystem and is not done twice.

## 0. Setup

This notebook supports Insomnia HPC. Request a compute node before running it, for example:

```bash
srun --pty -t 0-02:00 --gres=gpu:1 -c 8 --mem=64G -A edu /bin/bash
```

Run the notebook with the `ea_env` kernel. The Colab-only Google Drive setup is disabled below.
Llama 3.1 needs access approval on Hugging Face; paste your token in the login cell.

In [4]:
# Packages should be installed once in the Insomnia environment from a terminal:
# uv pip install --python ~/venvs/ea_env/bin/python \
#   "transformers==4.57.6" "datasets==3.6.0" "huggingface_hub<1.0" wandb \
#   jupyter ipywidgets

In [5]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less memory waste
import json, math, random, time
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

assert transformers.__version__ == "4.57.6", "Runtime > Restart session, then run from here."
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")

cuda Tesla T4


## 1. Settings

`PILOT = True` makes every number small. Change nothing else for the first run.

In [6]:
PILOT = True

TARGET = "meta-llama/Llama-3.1-8B-Instruct"
K = 8                  # tokens the drafter guesses per cycle
WIDTHS = [1.0, 0.75, 0.5]  # drafter feed-forward width, share of the original
DRAFT_VOCAB = 32000    # tokens the drafter can guess
N_TRAIN = 200 if PILOT else 3000  # prompts; the target writes the answers
EPOCHS = 1 if PILOT else 2
MAX_PROMPT, MAX_ANSWER = 256, 512  # token limits for the training data
ANCHORS = 64    # placeholder groups per training sequence (lower it if out of memory)
ACCUM = 4       # sequences per optimizer update
LR, WARMUP = 3e-4, 50  # peak learning rate; updates of warm-up
N_EVAL = 4 if PILOT else 20  # MT-Bench prompts for timing
MAX_NEW = 128 if PILOT else 256  # new tokens per prompt when timing
USE_WANDB = True  # log every run to Weights & Biases
# Colab: OUT = "/content/drive/MyDrive/peagle_ffn" + ("_pilot" if PILOT else "")
OUT = os.path.expanduser("~/peagle_ffn" + ("_pilot" if PILOT else ""))

In [7]:
from huggingface_hub import login
login()                                   # Llama 3.1 is gated: paste your token

# Colab-only setup:
# from google.colab import drive
# drive.mount("/content/drive")
os.makedirs(OUT, exist_ok=True)

if USE_WANDB:
    import wandb
    wandb.login()                         # paste your W&B key once

Mounted at /content/drive


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: einnuian (einnuian-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 2. The target model (frozen)

`target_step` runs the target on new tokens and returns three things:

- the **features**: the vector the target's output layer reads at each position (size 4,096);
- the **logits**: the target's score for every token, at each position;
- the **cache**: the stored keys and values, so the next call only processes new tokens.

In [9]:
from huggingface_hub import login
login()

In [10]:
tok = AutoTokenizer.from_pretrained(TARGET)
target = AutoModelForCausalLM.from_pretrained(TARGET, dtype=torch.bfloat16).to(DEV).eval()
target.requires_grad_(False)

cfg = target.config
D, FFN = cfg.hidden_size, cfg.intermediate_size      # 4096, 14336
HEADS, KV_HEADS = cfg.num_attention_heads, cfg.num_key_value_heads  # 32, 8
HEAD_DIM = D // HEADS                                # 128
EMBED = target.get_input_embeddings()                # frozen, shared with the drafter
eos = target.generation_config.eos_token_id
EOS = set(eos if isinstance(eos, list) else [eos])
PAD = tok.convert_tokens_to_ids("<|finetune_right_pad_id|>")
if PAD is None or PAD == tok.unk_token_id:
    PAD = tok.eos_token_id

@torch.no_grad()
def target_step(ids, cache):
    """ids: (1, n) new tokens. Returns features (1, n, D), logits (1, n, V), cache."""
    out = target.model(input_ids=ids, past_key_values=cache, use_cache=True)
    features = out.last_hidden_state
    return features, target.lm_head(features).float(), out.past_key_values

def chat_ids(text):
    return tok.apply_chat_template([{"role": "user", "content": text}],
                                   add_generation_prompt=True)

print(f"D = {D}, FFN = {FFN}, heads = {HEADS}, key/value heads = {KV_HEADS}, EOS = {EOS}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

... (3 outputs left out)

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1002.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 485.81 MiB is free. Including non-PyTorch memory, this process has 14.09 GiB memory in use. Of the allocated memory 13.98 GiB is allocated by PyTorch, and 7.49 MiB is reserved by PyTorch but unallocated.

## 3. Step 1: the target writes the training answers

The drafter must guess what **this** target writes, so the target writes the training answers
itself (greedy). The prompts are first user turns from UltraChat. Saved as one JSON line per
example; a restart continues where it stopped.

In [ ]:
DATA = f"{OUT}/train_data.jsonl"

def load_rows(path):
    return [json.loads(line) for line in open(path)] if os.path.exists(path) else []

def training_prompts(n):
    rows = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")
    rows = rows.shuffle(seed=0).select(range(3 * n))
    prompts = []
    for row in rows:
        ids = chat_ids(row["messages"][0]["content"])
        if len(ids) <= MAX_PROMPT:
            prompts.append(ids)
        if len(prompts) == n:
            break
    return prompts

@torch.no_grad()
def write_answers(prompts, batch=32):
    done = len(load_rows(DATA))
    for i in range(done, len(prompts), batch):
        chunk = prompts[i:i + batch]
        width = max(len(p) for p in chunk)
        x = torch.full((len(chunk), width), PAD, dtype=torch.long)
        mask = torch.zeros_like(x)
        for j, p in enumerate(chunk):  # pad on the left
            x[j, width - len(p):] = torch.tensor(p)
            mask[j, width - len(p):] = 1
        out = target.generate(input_ids=x.to(DEV), attention_mask=mask.to(DEV),
                              max_new_tokens=MAX_ANSWER, do_sample=False,
                              temperature=None, top_p=None, pad_token_id=PAD)
        with open(DATA, "a") as f:
            for j, p in enumerate(chunk):
                answer = []
                for t in out[j, width:].tolist():  # keep tokens up to and with the first EOS
                    answer.append(t)
                    if t in EOS:
                        break
                f.write(json.dumps({"prompt": p, "answer": answer}) + "\n")
        print(f"{min(i + batch, len(prompts))} / {len(prompts)} answers written")

if len(load_rows(DATA)) < N_TRAIN:
    write_answers(training_prompts(N_TRAIN))
data = load_rows(DATA)[:N_TRAIN]
lengths = [len(r["answer"]) for r in data]
print(len(data), "examples; mean answer length", round(sum(lengths) / len(lengths)), "tokens")
print(tok.decode(data[0]["answer"])[:400])

## 4. Step 2: the draft vocabulary

The drafter guesses only among the 32,000 most frequent tokens of the training answers. Its output
layer is these 32,000 rows of the target's output layer (frozen). This makes the drafter's output
layer 4 times smaller than the target's (128,256 rows). All three drafters share this vocabulary.

In [ ]:
counts = Counter(t for r in data for t in r["answer"])
keep = [t for t, _ in counts.most_common(DRAFT_VOCAB)]
keep += [t for t in EOS if t not in keep]  # the drafter must be able to guess EOS
D2T = torch.tensor(keep, device=DEV)       # draft id -> target id
T2D = torch.full((cfg.vocab_size,), -1, dtype=torch.long, device=DEV)
T2D[D2T] = torch.arange(len(keep), device=DEV)  # target id -> draft id (-1: none)
HEAD = target.lm_head.weight[D2T].detach().clone()  # (draft vocab, D), frozen

covered = sum(counts[t] for t in keep) / sum(counts.values())
print(f"{len(keep)} draft tokens cover {100 * covered:.2f}% of answer tokens")

## 5. The drafter

One drafter **slot** = one input vector. A slot for a real position `i` joins two vectors:
the target's feature at `i` and the embedding of the token at `i + 1`. It predicts the token
at `i + 2`. The slots after the last real one are **placeholders**: they get two learned vectors
(`h_shared` and `e_mask`) instead, and each predicts one more token ahead. All slots go through
4 decoder layers in one pass.

Only `ffn` changes between the three drafters.

In [ ]:
ROPE_BASE = 500000.0

def rope(x, pos):
    """Rotary position embedding. x: (1, heads, T, HEAD_DIM); pos: (T,)."""
    half = x.shape[-1] // 2
    freq = ROPE_BASE ** (-torch.arange(half, device=x.device, dtype=torch.float32) / half)
    angle = pos[:, None].float() * freq[None, :]
    cos, sin = angle.cos().to(x.dtype), angle.sin().to(x.dtype)
    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)

class RMSNorm(nn.Module):
    def __init__(self, size):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(size))

    def forward(self, x):
        x32 = x.float()
        x32 = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + 1e-5)
        return (x32 * self.weight.float()).to(x.dtype)

class Layer(nn.Module):
    """One Llama-style decoder layer: attention, then a gated feed-forward layer."""
    def __init__(self, ffn):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(D), RMSNorm(D)
        self.q = nn.Linear(D, HEADS * HEAD_DIM, bias=False)
        self.k = nn.Linear(D, KV_HEADS * HEAD_DIM, bias=False)
        self.v = nn.Linear(D, KV_HEADS * HEAD_DIM, bias=False)
        self.o = nn.Linear(HEADS * HEAD_DIM, D, bias=False)
        self.gate = nn.Linear(D, ffn, bias=False)  # these three are the part we make narrower
        self.up = nn.Linear(D, ffn, bias=False)
        self.down = nn.Linear(ffn, D, bias=False)

    def forward(self, x, pos, mask, cache=None):
        T = x.shape[1]
        h = self.norm1(x)
        q = self.q(h).view(1, T, HEADS, HEAD_DIM).transpose(1, 2)
        k = self.k(h).view(1, T, KV_HEADS, HEAD_DIM).transpose(1, 2)
        v = self.v(h).view(1, T, KV_HEADS, HEAD_DIM).transpose(1, 2)
        q, k = rope(q, pos), rope(k, pos)
        if cache is not None:  # cache = [keys, values] of earlier slots
            if cache[0] is not None:
                k, v = torch.cat([cache[0], k], 2), torch.cat([cache[1], v], 2)
            cache[0], cache[1] = k, v
        k = k.repeat_interleave(HEADS // KV_HEADS, dim=1)
        v = v.repeat_interleave(HEADS // KV_HEADS, dim=1)
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        x = x + self.o(a.transpose(1, 2).reshape(1, T, HEADS * HEAD_DIM))
        h = self.norm2(x)
        return x + self.down(F.silu(self.gate(h)) * self.up(h))

class Drafter(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.ffn = int(round(FFN * width / 128)) * 128  # 14336, 10752, 7168
        self.norm_h, self.norm_e = RMSNorm(D), RMSNorm(D)
        self.fc = nn.Linear(2 * D, D, bias=False)  # joins feature and token embedding
        self.h_shared = nn.Parameter(torch.randn(D) * 0.02)  # learned placeholder feature
        self.e_mask = nn.Parameter(torch.randn(D) * 0.02)  # learned placeholder embedding
        self.layers = nn.ModuleList([Layer(self.ffn) for _ in range(4)])
        self.norm_out = RMSNorm(D)
        for name, p in self.named_parameters():
            if p.dim() == 2:
                std = 0.02 / math.sqrt(8) if name.endswith(("o.weight", "down.weight")) else 0.02
                nn.init.normal_(p, std=std)

    def slots(self, features, tokens):
        """Real slots: feature i joined with the embedding of token i + 1."""
        auto = torch.is_autocast_enabled()
        dtype = features.dtype if auto else self.fc.weight.dtype
        features, e = features.to(dtype), EMBED(tokens).to(dtype)
        return self.fc(torch.cat([self.norm_h(features), self.norm_e(e)], dim=-1))

    def placeholder(self):
        both = torch.cat([self.norm_h(self.h_shared), self.norm_e(self.e_mask)])
        return self.fc(both)

    def run(self, x, pos, mask, cache=None):
        for i, layer in enumerate(self.layers):
            x = layer(x, pos, mask, None if cache is None else cache[i])
        return self.norm_out(x)

def logits_of(y):
    """Scores over the draft vocabulary."""
    return (y @ HEAD.to(y.dtype).T).float()

for w in WIDTHS:
    m = Drafter(w)
    n = sum(p.numel() for p in m.parameters())
    print(f"width {w:.2f}: ffn = {m.ffn}, {n / 1e6:.0f}M trained weights")
    del m

## 6. The two views of the drafter

**Training view.** One pass over a whole sequence. The real slots `0 .. L-2`, then `ANCHORS`
groups of `K - 1` placeholders. A group after anchor `n` acts as if `n` were the last real slot.
The **mask** says who may look at whom:

- a real slot looks at itself and earlier real slots;
- placeholder `s` of anchor `n` looks at real slots `0 .. n` and at placeholders `1 .. s` of the
  same group. It never looks at other groups.

**Decoding view.** Only the new real slots and `K - 1` placeholders go through the drafter. The
drafter's cache holds earlier real slots. After the pass, the placeholders are removed from the cache.

In [ ]:
def training_view(drafter, features, ids, answer_start, anchors):
    """features: (L, D) target features; ids: (L,) tokens. Returns logits and labels."""
    L, R, A = len(ids), len(ids) - 1, len(anchors)
    real = drafter.slots(features[:R], ids[1:])  # slot i: feature i + token i+1
    x = torch.cat([real, drafter.placeholder().expand(A * (K - 1), D)])[None]
    group_anchor = torch.tensor(anchors, dtype=torch.long, device=ids.device).repeat_interleave(K - 1)
    step = torch.arange(1, K, device=ids.device).repeat(A)  # 1 .. K-1 in each group
    pos = torch.cat([torch.arange(R, device=ids.device), group_anchor + step])

    T = R + A * (K - 1)
    mask = torch.zeros(T, T, dtype=torch.bool, device=ids.device)
    mask[:R, :R] = torch.ones(R, R, dtype=torch.bool, device=ids.device).tril()
    mask[R:, :R] = torch.arange(R, device=ids.device)[None, :] <= group_anchor[:, None]
    same_group = torch.arange(A, device=ids.device).repeat_interleave(K - 1)
    same = same_group[:, None] == same_group[None, :]
    mask[R:, R:] = same & (step[None, :] <= step[:, None])

    # labels: real slot i predicts token i+2; placeholder s of anchor n predicts token n+2+s
    targets = torch.cat([ids[2:], ids.new_zeros(1)]) if R > 0 else ids[:0]
    real_labels = torch.where(torch.arange(R, device=ids.device) + 2 < L, targets, -1)
    real_labels[: max(answer_start - 2, 0)] = -1  # learn only on the answer
    ph_labels = ids[group_anchor + 2 + step]
    labels = torch.cat([real_labels, ph_labels])
    labels = torch.where(labels >= 0, T2D[labels.clamp(min=0)], -1)  # draft ids; -1 = ignore

    y = drafter.run(x, pos, mask)
    return logits_of(y[0]), labels

def new_cache():
    return [[None, None] for _ in range(4)]

def cache_len(cache):
    return 0 if cache[0][0] is None else cache[0][0].shape[2]

def crop(cache, length):
    for c in cache:
        c[0], c[1] = c[0][:, :, :length], c[1][:, :, :length]

@torch.no_grad()
def draft(drafter, cache, features, next_tokens):
    """Adds real slots (features of new positions + the tokens after them) to the cache,
    runs K - 1 placeholders, and returns K guessed tokens (target ids)."""
    n, start = len(next_tokens), cache_len(cache)
    x = torch.cat([drafter.slots(features, next_tokens),
                   drafter.placeholder().expand(K - 1, D)])[None]
    pos = torch.arange(start, start + n + K - 1, device=x.device)
    q = torch.arange(n + K - 1, device=x.device)[:, None]
    keys = torch.arange(start + n + K - 1, device=x.device)[None, :]
    mask = keys <= start + q  # causal
    y = drafter.run(x, pos, mask, cache)
    crop(cache, start + n)                        # forget the placeholders
    guesses = logits_of(y[0, n - 1:]).argmax(-1)  # last real slot + K-1 placeholders
    return D2T[guesses]

## 7. Step 3: train the three drafters

Each training step: the target reads one example (no gradients), then the drafter predicts from
the target's features. Loss = cross-entropy on the target's own tokens, for real slots and
placeholders. The same examples, order and anchors for every width. A checkpoint goes to Drive
every 100 updates; a restart continues from it.

In [ ]:
import wandb

def anchors_for(ids_len, answer_start, seed):
    """Up to ANCHORS anchor positions whose K guesses all fall inside the answer."""
    low, high = max(answer_start - 2, 0), ids_len - K - 2
    if high < low:
        return []
    choices = list(range(low, high + 1))
    random.Random(seed).shuffle(choices)
    return sorted(choices[:ANCHORS])

def lr_at(update, total):
    warm = min(1.0, (update + 1) / WARMUP)
    return LR * warm * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * update / max(total, 1))))

def train(width):
    path = f"{OUT}/drafter_{int(width * 100)}.pt"
    torch.manual_seed(0)
    drafter = Drafter(width).to(DEV)
    opt = torch.optim.AdamW(drafter.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=0.0,
                            fused=DEV == "cuda")
    total = EPOCHS * len(data) // ACCUM
    update = 0
    if os.path.exists(path):
        state = torch.load(path, map_location=DEV)
        drafter.load_state_dict(state["drafter"])
        opt.load_state_dict(state["opt"])
        update = state["update"]
    if update >= total:
        print(f"width {width}: already trained")
        return drafter
    run = wandb.init(project="peagle-ffn", name=f"train_{int(width * 100)}", config=dict(
        width=width, K=K, n_train=len(data), epochs=EPOCHS, lr=LR, anchors=ANCHORS),
        mode="online" if USE_WANDB else "disabled", reinit=True)
    orders = [random.Random(e).sample(range(len(data)), len(data)) for e in range(EPOCHS)]
    start = time.time()
    while update < total:
        stats = Counter()
        for micro in range(ACCUM):
            k = update * ACCUM + micro  # the k-th example seen
            row = data[orders[k // len(data)][k % len(data)]]
            ids = torch.tensor(row["prompt"] + row["answer"], device=DEV)
            with torch.no_grad():
                features = target.model(input_ids=ids[None], use_cache=False).last_hidden_state[0]
            anchors = anchors_for(len(ids), len(row["prompt"]), seed=k)
            with torch.autocast(DEV, dtype=torch.bfloat16, enabled=DEV == "cuda"):
                logits, labels = training_view(drafter, features, ids, len(row["prompt"]), anchors)
            keep = labels >= 0
            loss = F.cross_entropy(logits[keep], labels[keep])
            (loss / ACCUM).backward()
            hit = (logits.argmax(-1) == labels) & keep
            n_real = len(ids) - 1
            stats["loss"] += loss.item() / ACCUM
            stats["real_hits"] += hit[:n_real].sum().item()
            stats["real_n"] += keep[:n_real].sum().item()
            stats["ph_hits"] += hit[n_real:].sum().item()
            stats["ph_n"] += keep[n_real:].sum().item()
        for g in opt.param_groups:
            g["lr"] = lr_at(update, total)
        torch.nn.utils.clip_grad_norm_(drafter.parameters(), 1.0)
        opt.step()
        opt.zero_grad(set_to_none=True)
        update += 1
        log = {"loss": stats["loss"], "lr": lr_at(update - 1, total),
               "acc_slot1": stats["real_hits"] / max(stats["real_n"], 1),  # guess 1
               "acc_placeholders": stats["ph_hits"] / max(stats["ph_n"], 1)}  # guesses 2..K
        run.log(log, step=update)
        if update % 25 == 0 or update == total:
            print(f"width {width}: update {update}/{total}  loss {log['loss']:.3f}  "
                  f"acc guess1 {log['acc_slot1']:.2f}  acc guesses2-{K} {log['acc_placeholders']:.2f}  "
                  f"{time.time() - start:.0f} s  "
                  f"{torch.cuda.max_memory_allocated() / 1e9 if DEV == 'cuda' else 0:.1f} GB peak")
        if update % 100 == 0 or update == total:
            torch.save({"drafter": drafter.state_dict(), "opt": opt.state_dict(),
                        "update": update}, path)
    run.finish()
    return drafter

for w in WIDTHS:
    train(w)
    torch.cuda.empty_cache()

## 8. Step 4: two checks before measuring

1. **Training view = decoding view.** For one example, the guesses from the training pass must
   equal the guesses from the decoding pass with its cache.
2. **Speculative decoding writes what the target writes.** With greedy decoding, every kept
   token is the target's own choice. In `bfloat16`, a pass over 9 tokens can round a little
   differently from 9 one-token passes, so a rare late difference is a rounding effect, not a bug.

In [ ]:
def load_drafter(width):
    m = Drafter(width).to(DEV)
    m.load_state_dict(torch.load(f"{OUT}/drafter_{int(width * 100)}.pt", map_location=DEV)["drafter"])
    return m.to(torch.bfloat16).eval()

drafters = {w: load_drafter(w) for w in WIDTHS}

In [ ]:
@torch.no_grad()
def check_views(drafter, row, anchor):
    ids = torch.tensor(row["prompt"] + row["answer"], device=DEV)
    features = target.model(input_ids=ids[None], use_cache=False).last_hidden_state[0]
    logits, _ = training_view(drafter, features, ids, len(row["prompt"]), [anchor])
    R = len(ids) - 1
    train_guess = D2T[torch.cat([logits[anchor:anchor + 1], logits[R:]]).argmax(-1)]
    cache = new_cache()  # decoding view, in two chunks
    half = anchor // 2
    draft(drafter, cache, features[:half], ids[1:half + 1])
    decode_guess = draft(drafter, cache, features[half:anchor + 1], ids[half + 1:anchor + 2])
    return train_guess.tolist(), decode_guess.tolist()

row = data[0]
anchor = len(row["prompt"]) + 10
for w, m in drafters.items():
    a, b = check_views(m, row, anchor)
    same = sum(x == y for x, y in zip(a, b))
    print(f"width {w}: {same}/{K} guesses equal  ({'OK' if same >= K - 1 else 'CHECK THE CODE'})")

## 9. Standard and speculative decoding (greedy, one request)

One **cycle** of speculative decoding:

1. **Draft.** The drafter guesses `K = 8` tokens in one pass.
2. **Verify.** The target reads the last kept token and the 8 guesses in one pass.
3. **Commit.** Keep guesses while they equal the target's choice; then add the target's own token.

In [ ]:
def sync():
    if DEV == "cuda":
        torch.cuda.synchronize()

@torch.no_grad()
def standard(prompt, max_new):
    _, logits, cache = target_step(torch.tensor([prompt], device=DEV), None)
    out = [logits[0, -1].argmax().item()]
    while len(out) < max_new and out[-1] not in EOS:
        _, logits, cache = target_step(torch.tensor([[out[-1]]], device=DEV), cache)
        out.append(logits[0, -1].argmax().item())
    return out, []

@torch.no_grad()
def speculative(prompt, drafter, max_new, clock=None):
    """Returns the new tokens and the number of accepted guesses in each cycle.
    clock: a Counter; if given, adds the time of the draft and verify parts (slower)."""
    ids = torch.tensor([prompt], device=DEV)
    features, logits, cache = target_step(ids, None)
    toks = torch.cat([ids[0], logits[0, -1:].argmax(-1)])  # all tokens so far
    feats, nexts = features[0], toks[1:]                   # real slots to add
    dcache, accepted = new_cache(), []
    while len(toks) - len(prompt) < max_new and toks[-1].item() not in EOS:
        if clock is not None: sync(); t0 = time.perf_counter()
        guesses = draft(drafter, dcache, feats, nexts)  # 1. draft K tokens
        if clock is not None: sync(); t1 = time.perf_counter()
        block = torch.cat([toks[-1:], guesses])[None]  # 2. verify in one pass
        features, logits, cache = target_step(block, cache)
        if clock is not None: sync(); t2 = time.perf_counter()
        choice = logits[0].argmax(-1)  # target's token after each prefix
        a = 0                          # 3. commit
        while a < K and guesses[a] == choice[a]:
            a += 1
        new = torch.cat([guesses[:a], choice[a:a + 1]])
        cache.crop(len(toks) + a)  # drop rejected positions
        feats, nexts = features[0, :a + 1], new
        toks = torch.cat([toks, new])
        accepted.append(a)
        if clock is not None:
            sync(); t3 = time.perf_counter()
            clock["draft"] += t1 - t0; clock["verify"] += t2 - t1; clock["other"] += t3 - t2
            clock["cycles"] += 1
    out = toks[len(prompt):].tolist()
    for i, t in enumerate(out):  # cut after the first EOS
        if t in EOS:
            out = out[:i + 1]
            break
    return out[:max_new], accepted

In [ ]:
mt = load_dataset("HuggingFaceH4/mt_bench_prompts", split="train")
eval_prompts = [chat_ids(r["prompt"][0]) for r in mt.select(range(N_EVAL))]

reference = [standard(p, MAX_NEW)[0] for p in eval_prompts]
for w, m in drafters.items():
    report = []
    for p, ref in zip(eval_prompts, reference):
        out, _ = speculative(p, m, MAX_NEW)
        first_diff = next((i for i, (x, y) in enumerate(zip(out, ref)) if x != y), None)
        report.append("same" if first_diff is None and len(out) == len(ref) else f"differs at {first_diff}")
    print(f"width {w}: " + ", ".join(report))
print("\nOne answer:\n", tok.decode(reference[0])[:300])

## 10. Step 5: measure

**Protocol (the same for every setup).** Two warm-up runs, then three timed passes over all
`N_EVAL` prompts; we report the mean. Then one more pass with a stopwatch on each part of the
cycle (this pass is slower, so it is not used for tokens per second).

**What each column means.**

- `tok/s`: new tokens per second. `speedup`: over standard decoding.
- `C`: mean committed tokens per cycle = 1 + mean accepted guesses.
- `a1 .. a8`: share of cycles in which the first `i` guesses were all accepted.
- `draft ms`, `verify ms`, `other ms`: mean time of each part of one cycle.

In [ ]:
def tokens_per_second(fn):
    fn(eval_prompts[0]); fn(eval_prompts[0])  # two warm-up runs
    speeds, accepted = [], []
    for _ in range(3):  # three timed passes
        sync(); t0 = time.perf_counter(); n = 0
        for p in eval_prompts:
            out, acc = fn(p)
            n += len(out)
            accepted += acc
        sync(); speeds.append(n / (time.perf_counter() - t0))
    return speeds, accepted

results = []
speeds, _ = tokens_per_second(lambda p: standard(p, MAX_NEW))
base = sum(speeds) / 3
results.append({"setup": "standard decoding", "tok/s": base, "spread": max(speeds) - min(speeds)})

for w, m in drafters.items():
    speeds, acc = tokens_per_second(lambda p: speculative(p, m, MAX_NEW))
    clock = Counter()
    for p in eval_prompts:
        speculative(p, m, MAX_NEW, clock)
    row = {"setup": f"P-EAGLE {int(w * 100)}%", "tok/s": sum(speeds) / 3,
           "spread": max(speeds) - min(speeds), "C": 1 + sum(acc) / len(acc)}
    for i in range(1, K + 1):
        row[f"a{i}"] = sum(a >= i for a in acc) / len(acc)
    for part in ["draft", "verify", "other"]:
        row[f"{part} ms"] = 1000 * clock[part] / clock["cycles"]
    results.append(row)

print(f"{'setup':18}{'tok/s':>8}{'±':>6}{'speedup':>9}{'C':>6}{'a1':>6}{'a2':>6}{'a4':>6}{'a8':>6}"
      f"{'draft ms':>10}{'verify ms':>10}{'other ms':>10}")
for r in results:
    line = f"{r['setup']:18}{r['tok/s']:8.1f}{r['spread']:6.1f}{r['tok/s'] / base:9.2f}"
    if "C" in r:
        line += f"{r['C']:6.2f}{r['a1']:6.2f}{r['a2']:6.2f}{r['a4']:6.2f}{r['a8']:6.2f}"
        line += f"{r['draft ms']:10.2f}{r['verify ms']:10.2f}{r['other ms']:10.2f}"
    print(line)
json.dump(results, open(f"{OUT}/results.json", "w"), indent=1)

## 11. Read the result with the speed equation

Time per token = (draft + verify + other) / C. A narrower drafter is faster end to end exactly
when its relative saving in cycle time is bigger than its relative loss in C. The cell prints
both numbers for each narrow drafter against the 100% drafter, and the speed the equation predicts.

In [ ]:
ref = next(r for r in results if r["setup"] == "P-EAGLE 100%")
ref_cycle = ref["draft ms"] + ref["verify ms"] + ref["other ms"]
for r in results:
    if "C" not in r:
        continue
    cycle = r["draft ms"] + r["verify ms"] + r["other ms"]
    print(f"{r['setup']:14}  cycle {cycle:6.2f} ms ({100 * (cycle / ref_cycle - 1):+5.1f}%)  "
          f"C {r['C']:.2f} ({100 * (r['C'] / ref['C'] - 1):+5.1f}%)  "
          f"drafter {r['draft ms']:.2f} ms ({100 * (r['draft ms'] / ref['draft ms'] - 1):+5.1f}%)  "
          f"predicted {1000 * r['C'] / cycle:6.1f} tok/s")

if USE_WANDB:
    run = wandb.init(project="peagle-ffn", name="eval", reinit=True,
                     config={"K": K, "n_eval": N_EVAL, "max_new": MAX_NEW,
                             "gpu": torch.cuda.get_device_name(0) if DEV == "cuda" else "cpu"})
    cols = list(results[-1].keys())
    run.log({"results": wandb.Table(columns=cols, data=[[r.get(c) for c in cols] for r in results])})
    run.finish()

## What to report

1. **The table.** Tokens per second and speedup over standard decoding for all three widths;
   C and a1..a8; the time of each part of the cycle.
2. **The two checks** (same guesses in both views; same output as standard decoding).
3. **The speed-equation lines**: how much cycle time each narrow drafter saved, and how much C it lost.

The guide (Part 3) explains how to read these numbers and what counts as a positive result.